# Silver Transform — users_hist
**Proyecto 2 FHBD | Capa Silver — Ejecución Manual**

Equivalente manual del Task 2 del DAG.

Lee users 2019 + 2020 + 2021 desde Bronze y consolida en
`nessie.silver.users_hist` usando **MERGE/upsert** en Iceberg.

---
⚠️ Ejecutar este notebook si el Task 2 del DAG falla.

**Pre-requisito:** Bronze debe tener los 3 años de users cargados.

## 1. Configuración

In [ ]:
import os

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT',  'http://proyecto2-minio:9000')
MINIO_ACCESS   = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET   = os.getenv('MINIO_SECRET_KEY', 'minioadmin123')
BRONZE_BUCKET  = os.getenv('MINIO_BUCKET_BRONZE', 'bronze')
NESSIE_URL     = os.getenv('NESSIE_URL', 'http://proyecto2-nessie:19120/api/v1')
NESSIE_BRANCH  = os.getenv('NESSIE_BRANCH', 'main')
SILVER_NS      = os.getenv('SILVER_NAMESPACE', 'silver')
SPARK_MASTER   = os.getenv('SPARK_MASTER', 'spark://spark-master:7077')
YEARS          = [2019, 2020, 2021]

print(f'MinIO     : {MINIO_ENDPOINT}')
print(f'Nessie    : {NESSIE_URL}')
print(f'Namespace : nessie.{SILVER_NS}')
print(f'Spark     : {SPARK_MASTER}')
print(f'Años      : {YEARS}')

## 2. Verificar archivos Bronze en MinIO

In [ ]:
import boto3
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS,
    aws_secret_access_key=MINIO_SECRET,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1',
)

print('Verificando archivos Bronze de users:')
for year in YEARS:
    key = f'users/{year}/users_{year}.parquet'
    try:
        obj = s3.head_object(Bucket=BRONZE_BUCKET, Key=key)
        size_mb = obj['ContentLength'] / 1024 / 1024
        print(f'  ✅ s3://{BRONZE_BUCKET}/{key} ({size_mb:.2f} MB)')
    except Exception:
        print(f'  ❌ FALTA: s3://{BRONZE_BUCKET}/{key}')

## 3. Crear SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('silver_users_hist_notebook')
    .master(SPARK_MASTER)
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,'
            'org.projectnessie.spark.extensions.NessieSparkSessionExtensions')
    .config('spark.sql.catalog.nessie',              'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog')
    .config('spark.sql.catalog.nessie.uri',          NESSIE_URL)
    .config('spark.sql.catalog.nessie.ref',          NESSIE_BRANCH)
    .config('spark.sql.catalog.nessie.authentication.type', 'NONE')
    .config('spark.sql.catalog.nessie.warehouse',    's3a://iceberg/')
    .config('spark.hadoop.fs.s3a.endpoint',          MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key',        MINIO_ACCESS)
    .config('spark.hadoop.fs.s3a.secret.key',        MINIO_SECRET)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl',              'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)

print(f'✅ SparkSession creada — versión {spark.version}')

## 4. Leer Bronze y unir los 3 años

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
from datetime import datetime

dfs = []
for year in YEARS:
    path = f's3a://{BRONZE_BUCKET}/users/{year}/users_{year}.parquet'
    try:
        df = spark.read.parquet(path)
        df = df.withColumn('anio', F.lit(year))
        count = df.count()
        dfs.append(df)
        print(f'  ✅ users {year}: {count:,} filas')
    except Exception as e:
        print(f'  ❌ Error leyendo {path}: {e}')

df_all = dfs[0]
for df in dfs[1:]:
    df_all = df_all.unionByName(df, allowMissingColumns=True)

print(f'\nTotal combinado: {df_all.count():,} filas')

## 5. Calidad de datos y normalización

In [ ]:
# Calidad: verificar nulls antes
print('Nulls por columna (antes de limpieza):')
df_all.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ['Id', 'DisplayName', 'Reputation', 'AccountId']
]).show()

# Eliminar duplicados por Id
total_antes = df_all.count()
df_dedup    = df_all.dropDuplicates(['Id'])
total_dedup = df_dedup.count()
print(f'Duplicados eliminados: {total_antes - total_dedup:,}')

# Normalizar tipos y agregar metadata Silver
fecha_cargue = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

df_silver = (
    df_dedup
    .withColumn('Id',         F.col('Id').cast('long'))
    .withColumn('Reputation', F.col('Reputation').cast('long'))
    .withColumn('Views',      F.col('Views').cast('long'))
    .withColumn('UpVotes',    F.col('UpVotes').cast('long'))
    .withColumn('DownVotes',  F.col('DownVotes').cast('long'))
    .withColumn('AccountId',  F.col('AccountId').cast('long'))
    .withColumn('fecha_cargue',
                F.lit(fecha_cargue).cast(TimestampType()))
    .withColumn('fuente', F.lit('clickhouse_stackoverflow'))
    .fillna({'DisplayName': 'Unknown', 'Location': '', 'AccountId': -1})
)

# Calidad: verificar nulls después
print('\nNulls por columna (después de limpieza):')
df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ['Id', 'DisplayName', 'Reputation', 'AccountId']
]).show()

print(f'\n✅ DataFrame Silver listo: {df_silver.count():,} filas')
df_silver.printSchema()

## 6. Crear namespace y tabla Iceberg

In [ ]:
# Crear namespace silver si no existe
try:
    spark.sql(f'CREATE NAMESPACE IF NOT EXISTS nessie.{SILVER_NS}')
    print(f'✅ Namespace nessie.{SILVER_NS} listo')
except Exception as e:
    print(f'Namespace: {e}')

# Crear tabla si no existe
full_table = f'nessie.{SILVER_NS}.users_hist'
try:
    spark.sql(f'DESCRIBE TABLE {full_table}')
    print(f'✅ Tabla {full_table} ya existe')
except Exception:
    print(f'Creando tabla {full_table}...')
    df_silver.limit(0).writeTo(full_table) \
        .tableProperty('write.format.default', 'parquet') \
        .createOrReplace()
    print(f'✅ Tabla {full_table} creada')

## 7. MERGE/upsert en Iceberg

In [ ]:
df_silver.createOrReplaceTempView('users_updates')

spark.sql(f"""
    MERGE INTO {full_table} AS target
    USING users_updates AS source
    ON target.Id = source.Id
    WHEN MATCHED THEN
        UPDATE SET
            target.Reputation     = source.Reputation,
            target.LastAccessDate = source.LastAccessDate,
            target.DisplayName    = source.DisplayName,
            target.Views          = source.Views,
            target.UpVotes        = source.UpVotes,
            target.DownVotes      = source.DownVotes,
            target.Location       = source.Location,
            target.anio           = source.anio,
            target.fecha_cargue   = source.fecha_cargue,
            target.fuente         = source.fuente
    WHEN NOT MATCHED THEN
        INSERT *
""")

total = spark.sql(f'SELECT COUNT(*) as c FROM {full_table}').collect()[0]['c']
print(f'✅ MERGE completado')
print(f'   {full_table} → {total:,} filas')

## 8. Verificación final

In [ ]:
print('Distribución por año:')
spark.sql(f"""
    SELECT anio, COUNT(*) as total
    FROM {full_table}
    GROUP BY anio
    ORDER BY anio
""").show()

print('Muestra de datos:')
spark.sql(f'SELECT Id, DisplayName, Reputation, anio, fecha_cargue FROM {full_table} LIMIT 5').show()

print('Historial de snapshots Iceberg:')
spark.sql(f'SELECT * FROM {full_table}.history').show()

spark.stop()
print('\n✅ Silver users_hist completado')